In [9]:
from pathlib import Path
import sys
import warnings
import re 
import html 

import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path("..").resolve()
DATA_DIR = PROJECT_ROOT / "data" 
RAW_DATA_PATH = DATA_DIR / "comments.csv"
CLEANED_DATA_PATH = DATA_DIR / "cleaned_comments.csv" # Output file location after preprocessing.
REQUIRED_COLUMNS = ["CommentText", "Sentiment"] # Useful columns 
RANDOM_STATE = 42

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

warnings.filterwarnings("ignore")

from utils import load_raw_data

# Check for langdetect
try: 
    from langdetect import detect, DetectorFactory 
except ImportError as exc: 
    raise ImportError(
        "Install langdetect before running this notebook: pip install langdetect"
    ) from exc


import nltk
from nltk.corpus import stopwords 
from nltk.stem import WordNetLemmatizer 

nltk.download("stopwords", quiet=True)
nltk.download("wordnet", quiet=True)
nltk.download("omw-1.4", quiet=True)


True

 ------------------------------------------------------------------
 1. Load raw data and keep the fields used for modeling
 ------------------------------------------------------------------


In [10]:
df = load_raw_data(RAW_DATA_PATH) 

df = df[REQUIRED_COLUMNS].copy()
df["CommentText"] = df["CommentText"].astype("string").str.strip()
df["Sentiment"] = df["Sentiment"].astype("string").str.strip()

print("Raw modeling shape:", df.shape)
print("Missing values before cleanup:")
print(df.isnull().sum())

before = len(df) 
df = df.dropna(subset=REQUIRED_COLUMNS) 
df = df[df["CommentText"].str.len() > 0].reset_index(drop=True) 

print(f"Dropped {before - len(df)} rows with missing/blank required values")
print("Shape after basic cleanup:", df.shape)
print("Sentiment distribution:")
print(df["Sentiment"].value_counts())


Raw modeling shape: (103482, 2)
Missing values before cleanup:
CommentText    0
Sentiment      0
dtype: int64
Dropped 0 rows with missing/blank required values
Shape after basic cleanup: (103482, 2)
Sentiment distribution:
Sentiment
Negative    35073
Positive    34308
Neutral     34101
Name: count, dtype: Int64


 ------------------------------------------------------------------
 2. Remove duplicate comments
 ------------------------------------------------------------------


In [11]:
before = len(df)

df["CommentText"] = df["CommentText"].str.lower()
df = df.drop_duplicates(subset=["CommentText"]).reset_index(drop=True)

print(f"Dropped {before - len(df)} duplicate comments -> shape {df.shape}")
print("Sentiment distribution after de-duplication:")
print(df["Sentiment"].value_counts())


Dropped 2584 duplicate comments -> shape (100898, 2)
Sentiment distribution after de-duplication:
Sentiment
Negative    34752
Neutral     33259
Positive    32887
Name: count, dtype: Int64


 ------------------------------------------------------------------
 3. Detect comment language
 ------------------------------------------------------------------


DetectorFactory is used to make the language detection deterministic so the same text always yields the same language prediction.

In [12]:
DetectorFactory.seed = RANDOM_STATE 

# Create a function to detect the language of a given text.
def detect_language(text: str) -> str:
    text = str(text).strip() 
    
    if len(text) < 3:  
        return "unknown"
    try:
        return detect(text) 
    except Exception:
        return "unknown"


df["Language"] = df["CommentText"].apply(detect_language) 

print("Detected language counts:")
print(df["Language"].value_counts().head(10))
df[["CommentText", "Language"]].head()


Detected language counts:
Language
en         89816
af           859
tl           785
so           767
fr           667
cy           605
no           599
unknown      587
it           584
da           502
Name: count, dtype: int64


,CommentText,Language
0,anyone know what movie this is?,en
1,the fact they're holding each other back while...,en
2,waiting next video will be?,en
3,thanks for the great video. i don't understand...,en
4,good person helping good people. this is how i...,en


In [13]:
before = len(df)
df_en = df.loc[df["Language"].eq("en")].reset_index(drop=True) 
after = len(df_en)

print(f"Kept {after} / {before} comments ({after / before:.1%}) after English-only filter")
print(f"Dropped {before - after} non-English/unknown comments")
print("Sentiment distribution after language filter:")
print(df_en["Sentiment"].value_counts())


Kept 89816 / 100898 comments (89.0%) after English-only filter
Dropped 11082 non-English/unknown comments
Sentiment distribution after language filter:
Sentiment
Negative    32272
Positive    29411
Neutral     28133
Name: count, dtype: Int64


## ⭐️⭐️ MOST IMPORTANT PRE PROCESSING FEATURE 

 ------------------------------------------------------------------
 4. Define the text-cleaning function
- lowercase
- remove URLs, HTML tags, @mentions
- preserve negation in contractions such as don't/can't
- remove numbers/punctuation/emoji
- remove stopwords (keeping negations like *no/not/nor*)
- lemmatize tokens
 ------------------------------------------------------------------


In [14]:
stop_words = set(stopwords.words("english")) 
stop_words -= {"no", "not", "nor"}  # keep negations, they carry sentiment . Remove three words from stopwords
lemmatizer = WordNetLemmatizer()  

URL_RE = re.compile(r"https?://\S+|www\.\S+") 
HTML_RE = re.compile(r"<.*?>")
MENTION_RE = re.compile(r"@\w+") 
NON_ALPHA_RE = re.compile(r"[^a-z\s]") 
MULTISPACE_RE = re.compile(r"\s+")

# sub means subsitute 
def clean_text(text: str) -> str:
    text = html.unescape(str(text).lower()) 
    text = re.sub(r"\b(can't|cannot)\b", "can not", text) 
    text = re.sub(r"n't\b", " not", text) 
    text = URL_RE.sub(" ", text) 
    text = HTML_RE.sub(" ", text) 
    text = MENTION_RE.sub(" ", text) 
    text = NON_ALPHA_RE.sub(" ", text) 
    text = MULTISPACE_RE.sub(" ", text).strip() 

    tokens = [] 
    for t in text.split():  
        if t not in stop_words and len(t) > 1: 
            tokens.append(lemmatizer.lemmatize(t))

    return " ".join(tokens)   

In [15]:
df_clean = df_en.copy() 
df_clean["clean_text"] = df_clean["CommentText"].apply(clean_text) 

before = len(df_clean)
df_clean = df_clean[df_clean["clean_text"].str.len() > 0].reset_index(drop=True) 

print(f"Dropped {before - len(df_clean)} rows that became empty after cleaning")
print("Final shape:", df_clean.shape)
print("Final sentiment distribution:")
print(df_clean["Sentiment"].value_counts())

df_clean = df_clean[["CommentText", "Sentiment", "Language", "clean_text"]]
df_clean.to_csv(CLEANED_DATA_PATH, index=False) 
print(f"Saved cleaned dataset to {CLEANED_DATA_PATH}")

df = df_clean


Dropped 138 rows that became empty after cleaning
Final shape: (89678, 4)
Final sentiment distribution:
Sentiment
Negative    32252
Positive    29399
Neutral     28027
Name: count, dtype: Int64
Saved cleaned dataset to /Users/tiyasharma/Desktop/TIYA/AI ML/Project /Comments Sentimental Analysis  copy 2/data/cleaned_comments.csv


In [16]:
print("Examples:\n")
for i in range(min(5, len(df))):
    print("RAW  :", df["CommentText"].iloc[i][:120])
    print("CLEAN:", df["clean_text"].iloc[i][:120])
    print("-" * 60)


Examples:

RAW  : anyone know what movie this is?
CLEAN: anyone know movie
------------------------------------------------------------
RAW  : the fact they're holding each other back while equally being most aggressive
CLEAN: fact holding back equally aggressive
------------------------------------------------------------
RAW  : waiting next video will be?
CLEAN: waiting next video
------------------------------------------------------------
RAW  : thanks for the great video. i don't understand why the db continues to be accesible through port 8080 when the local mac
CLEAN: thanks great video not understand db continues accesible port local machine connects docker container port not possible 
------------------------------------------------------------
RAW  : good person helping good people. this is how it is in america with the exception of ny and dc.
CLEAN: good person helping good people america exception ny dc
------------------------------------------------------------
